In [ ]:
import os
import json
import uuid
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal
from getpass import getpass
from pprint import pprint
from dotenv import load_dotenv
from supabase import create_client

# AX_TeamDB 폴더를 열고 실행한다. 경로가 다르면 여기서 먼저 멈춘다.
project_dir = Path.cwd()
if not (project_dir / "users.ipynb").exists():
    raise RuntimeError("AX_TeamDB 폴더에서 이 노트북을 실행하세요.")
load_dotenv(project_dir / ".env", override=True)
url = os.getenv("SUPABASE_URL", "").rstrip("/")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY", "")
if not url or not key or "/rest/v1" in url:
    raise ValueError(".env의 프로젝트 기본 URL과 공개 API 키를 확인하세요.")
supabase = create_client(url, key)
user_id = None  # 연결을 새로 만들었으므로 이전 로그인 ID를 재사용하지 않는다.

# 같은 커널에서 준비 셀을 다시 눌러도 진행 중인 주문 ID는 버리지 않는다.
# 주문 도중 커널을 재시작하면 이 기억도 사라진다. 삭제 전 ID를 확인해 둔다.
demo_order_id = globals().get("demo_order_id")
print("연결 객체 준비 완료. 아직 로그인하거나 데이터를 저장하지 않았습니다.")

In [ ]:
# 세 파일은 클래스 전체가 하나의 코드 셀에 들어 있음을 확인했다.
for filename, class_name in [
    ("users.ipynb", "UserService"),
    ("orders.ipynb", "Order_service"),
    ("order_items.ipynb", "OrderItemService"),
]:
    notebook = json.loads((project_dir / filename).read_text(encoding="utf-8"))
    class_cells = [
        "".join(cell["source"])
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
        and "".join(cell["source"]).lstrip().startswith(f"class {class_name}:")
    ]
    if len(class_cells) != 1:
        raise RuntimeError(f"{filename}의 클래스 셀 구성이 달라졌습니다.")
    exec(class_cells[0], globals())

# 기존 일부 메서드는 전역 supabase를 사용하므로 위에서 만든 하나의 객체를 유지한다.
user_service = UserService()
order_service = Order_service(supabase)
order_item_service = OrderItemService(supabase)
print("기존 회원, 주문, 주문상품 클래스 준비 완료")

In [ ]:
PREPARE_PRODUCT = False
NEW_SELLER_ACCOUNT = False

if PREPARE_PRODUCT:
    seller_email = input("판매자 이메일: ").strip()
    seller_password = getpass("판매자 비밀번호: ")
    try:
        if NEW_SELLER_ACCOUNT:
            signup = supabase.auth.sign_up({"email": seller_email, "password": seller_password})
            if signup.session is None:
                raise RuntimeError("가입 이메일을 확인한 뒤 NEW_SELLER_ACCOUNT=False로 다시 실행하세요.")
        seller_login = supabase.auth.sign_in_with_password({"email": seller_email, "password": seller_password})
        if seller_login.user is None or seller_login.session is None:
            raise RuntimeError("판매자 로그인에 실패했습니다.")
        seller_id = seller_login.user.id
        seller_details = supabase.table("user_details").select("*").eq("id", seller_id).execute().data
        if not seller_details:
            seller_details = user_service.create_user_detail(seller_id, type="SELLER")
        if not seller_details or seller_details[0]["type"] != "SELLER" or seller_details[0]["deleted_at"] is not None:
            raise PermissionError("활성 SELLER 계정이 필요합니다. 기존 회원 유형은 변경하지 않습니다.")

        # 기존 상품 클래스는 수정하지 않고 이 준비 셀에서 직접 등록한다.
        prepared_product = supabase.table("products").insert({
            "seller_id": seller_id,
            "name": "EX 발표용 머그컵",
            "price": 12000,
            "description": "주문 연결을 보여주기 위한 실습용 상품",
        }).execute().data
        if not prepared_product:
            raise RuntimeError("상품 응답이 비었습니다. Table Editor에서 저장 여부를 먼저 확인하세요.")
        pprint([{k: row[k] for k in ("id", "name", "price")} for row in prepared_product])
    finally:
        seller_password = None
        supabase.auth.sign_out()
else:
    print("판매자 준비를 건너뜁니다. 기존 상품으로 진행할 수 있습니다.")

In [ ]:
CREATE_ACCOUNT = False  # 처음 계정을 만들 때만 True
email = input("구매자 이메일: ").strip()
password = getpass("구매자 비밀번호: ")
if not email or not password:
    raise ValueError("이메일과 비밀번호를 입력하세요.")

if CREATE_ACCOUNT:
    signup = supabase.auth.sign_up({"email": email, "password": password})
    if signup.session is None:
        password = None
        raise RuntimeError("가입 이메일을 확인한 뒤 CREATE_ACCOUNT=False로 이 셀부터 다시 실행하세요.")
    print("가입 요청 완료. 다음 셀에서 로그인을 확인합니다.")
else:
    print("기존 계정으로 로그인할 준비가 되었습니다.")

In [ ]:
# 로그인 재시도가 실패했을 때 이전 사용자 ID로 진행하지 않도록 먼저 비운다.
user_id = None
try:
    login = supabase.auth.sign_in_with_password({"email": email, "password": password})
    if login.user is None or login.session is None:
        raise RuntimeError("로그인이 완료되지 않았습니다.")
    current_user = supabase.auth.get_user()
    if current_user.user is None:
        raise RuntimeError("현재 로그인 사용자를 확인할 수 없습니다.")
    user_id = current_user.user.id
finally:
    password = None

print("로그인 성공. 이후 주문에는 현재 사용자의 ID가 들어갑니다.")

In [ ]:
if not user_id:
    raise RuntimeError("2단계 로그인을 먼저 완료하세요.")

details = supabase.table("user_details").select("*").eq("id", user_id).execute().data
if not details:
    details = user_service.create_user_detail(user_id, type="BUYER")
if not details or details[0]["deleted_at"] is not None:
    raise RuntimeError("활성 회원 상세 정보가 필요합니다. 저장 여부와 권한을 확인하세요.")
if details[0]["type"] != "BUYER":
    raise PermissionError("본 시연은 BUYER 계정으로 진행합니다. 구매자 계정으로 로그인하세요.")

# GUI의 배송지 입력창을 대신한다. 필요하면 발표용 값만 수정한다.
delivery = {"zipcode": "00000", "address": "발표용 가상 주소", "address_sub": "실습실"}
updated_details = user_service.update_user_detail(user_id, **delivery)
if not updated_details:
    raise RuntimeError("회원 정보 수정 결과가 없습니다. 현재 계정의 RLS 권한을 확인하세요.")
print("회원 상세 정보 준비 완료: BUYER / 발표용 배송지")

In [ ]:
products = (supabase.table("products")
    .select("id,name,price")
    .is_("deleted_at", "null")
    .order("name")
    .limit(20)
    .execute().data)
if not products:
    raise RuntimeError("보이는 상품이 없습니다. 선택 단계 S에서 상품을 준비하거나 상품 SELECT 정책을 확인하세요.")
for number, product in enumerate(products, start=1):
    print(f"{number}. {product['name']} / {product['price']}원 / ID={product['id']}")

In [ ]:
product_number = int(input("주문할 상품 번호: "))
quantity = 2  # GUI의 수량 입력창. 1, 2, 3 등 양의 정수로 수정한다.
if not 1 <= product_number <= len(products):
    raise ValueError("목록에 있는 상품 번호를 선택하세요.")
if type(quantity) is not int or quantity <= 0:
    raise ValueError("수량은 1 이상의 정수여야 합니다.")
selected_product_id = products[product_number - 1]["id"]
print("선택 완료. 아직 주문을 저장하지 않았습니다.")

In [ ]:
if not user_id:
    raise RuntimeError("먼저 로그인하세요.")
if demo_order_id is not None:
    raise RuntimeError("이 커널에 진행 중인 주문이 있습니다. 6단계 조회와 7단계 삭제를 먼저 실행하세요.")

selected = (supabase.table("products").select("id,name,price")
    .eq("id", selected_product_id).is_("deleted_at", "null").execute().data)
if len(selected) != 1:
    raise RuntimeError("선택한 상품이 삭제되었거나 조회할 수 없습니다.")
product = selected[0]
unit_price = Decimal(str(product["price"]))
if not unit_price.is_finite() or unit_price < 0 or unit_price != unit_price.to_integral_value():
    raise ValueError("이 예제는 0 이상의 원 단위 정수 가격만 사용합니다.")
if type(quantity) is not int or quantity <= 0:
    raise ValueError("수량은 1 이상의 정수여야 합니다.")
total_price = int(unit_price) * quantity

order_data = {
    "order_no": f"EX-{uuid.uuid4().hex}",  # NOT NULL + UNIQUE인 주문번호
    "order_name": "발표용 구매자",        # 기본값 없는 필수 표시명
    "user_id": user_id,
    "total_price": total_price,
    "order_status": "PENDING",           # 예제에서 정한 주문 접수 상태
    **delivery,
}
created_orders = order_service.create_order(order_data)
if not created_orders:
    raise RuntimeError("주문 응답이 비었습니다. 재시도 전에 Table Editor에서 저장 여부를 확인하세요.")
demo_order_id = created_orders[0]["id"]  # 부모 ID를 받아 자식 행에 전달한다.
print("생성한 주문 ID:", demo_order_id)

created_items = order_item_service.create_order_item(
    order_id=demo_order_id,
    product_id=product["id"],
    item_name=product["name"],
    item_price=int(unit_price),
    quantity=quantity,
)
if not created_items:
    raise RuntimeError("주문상품 응답이 비었습니다. 6단계에서 확인하고 필요하면 7단계로 정리하세요.")
print(f"주문 완료: {product['name']} × {quantity}개 = {total_price:,}원")
pprint(created_items)

In [ ]:
if not demo_order_id or not user_id:
    raise RuntimeError("로그인과 주문 생성이 먼저 필요합니다.")
order_view = (supabase.table("orders")
    .select("id,order_no,order_name,user_id,total_price,order_status,deleted_at,order_items(id,order_id,product_id,item_name,item_price,quantity,deleted_at,products(id,name,price))")
    .eq("id", demo_order_id)
    .eq("user_id", user_id)
    .is_("deleted_at", "null")
    .is_("order_items.deleted_at", "null")
    .execute().data)
if not order_view:
    raise RuntimeError("현재 사용자에게 보이는 활성 주문이 없습니다. 주문 ID와 권한을 확인하세요.")
pprint(order_view)

In [ ]:
if not demo_order_id or not user_id:
    raise RuntimeError("삭제할 주문 ID와 로그인 사용자가 필요합니다.")

# order_items에는 user_id가 없으므로 먼저 부모 주문이 내 것인지 확인한다.
owned_order = (supabase.table("orders").select("id,deleted_at")
    .eq("id", demo_order_id).eq("user_id", user_id).execute().data)
if not owned_order:
    raise PermissionError("내 주문인지 확인할 수 없어 삭제하지 않습니다.")

active_items = order_item_service.get_order_items(demo_order_id)
for item in active_items:
    deleted_items = order_item_service.soft_delete_order_item(item["id"])
    if not deleted_items:
        raise RuntimeError("주문상품 삭제가 확인되지 않았습니다. 권한/저장 결과 확인 후 다시 실행하세요.")

if owned_order[0]["deleted_at"] is None:
    deleted_orders = order_service.delete_order(user_id, demo_order_id)
    if not deleted_orders:
        raise RuntimeError("부모 주문 삭제가 확인되지 않았습니다. 주문 ID를 유지하고 다시 확인하세요.")

print("주문과 주문상품의 소프트 삭제 요청을 마쳤습니다. 다음 셀에서 결과를 확인합니다.")

In [ ]:
if not demo_order_id or not user_id:
    raise RuntimeError("검증할 주문 ID와 로그인 사용자가 필요합니다.")
deleted_order = (supabase.table("orders").select("id,deleted_at")
    .eq("id", demo_order_id).eq("user_id", user_id).execute().data)
active_items = order_item_service.get_order_items(demo_order_id)
deleted_items = order_item_service.get_deleted_order_items(demo_order_id)
if not deleted_order or deleted_order[0]["deleted_at"] is None or active_items:
    raise RuntimeError("삭제 검증이 끝나지 않았습니다. 7단계 결과와 DB 권한을 확인하세요.")

print("부모 주문 삭제 시각:", deleted_order[0]["deleted_at"])
print("남은 활성 주문상품:", len(active_items))
print("삭제 기록으로 남은 주문상품:", len(deleted_items))

# 주문상품 생성이 실패한 주문을 정리했다면 deleted_items는 0개일 수 있다.
# 정상 시연에서는 위에서 삭제한 주문상품의 product_id로 원본 상품을 확인한다.
for item in deleted_items:
    original = (supabase.table("products").select("id,name,deleted_at")
        .eq("id", item["product_id"]).execute().data)
    if not original or original[0]["deleted_at"] is not None:
        raise RuntimeError("상품 원본을 활성 상태로 확인할 수 없습니다. DB 상태와 권한을 확인하세요.")
    print("상품 원본 유지:", original[0]["name"])

last_demo_order_id = demo_order_id
demo_order_id = None
print("삭제 결과 확인 완료. 다음 주문 예제를 실행할 수 있습니다.")

In [ ]:
supabase.auth.sign_out()
user_id = None
password = None
print("로그아웃 완료. 다시 주문하려면 1~3단계부터 로그인과 회원 정보를 준비하세요.")